# Arquitetura do Pipeline de Dados — Qualidade e Observabilidade

## 1. Visão Geral

Este documento descreve a arquitetura completa do pipeline de dados do projeto **Arquitetura_TechChallenge**, com foco em **qualidade de dados** e **observabilidade/monitoramento**. O pipeline segue o padrão **medalhão** (Bronze → Silver → Gold) para dados do INEP — Avaliação Nacional de Alfabetização (ANA), com fonte no BigQuery via Base dos Dados.

### 1.1 Diagrama de Arquitetura

```
┌─────────────────────────────────────────────────────────────────────────────────┐
│                           FONTE DE DADOS EXTERNA                                 │
│                         BigQuery (Base dos Dados)                                │
└───────────────────────────────┬─────────────────────────────────────────────────┘
                                │  (SELECT * via google-cloud-bigquery)
                                ▼
┌─────────────────────────────────────────────────────────────────────────────────┐
│  ORQUESTRAÇÃO: AWS Step Functions → 7 Jobs do Databricks (Lakeflow Jobs)        │
│                                                                                 │
│  Job 01: Ingestão Bronze    │  Job 02: DQ Bronze  │  Job 03: Tratamento Silver   │
│  Job 04: DQ Silver          │  Job 05: Dim Gold   │  Job 06: Fato Gold           │
│  Job 07: DQ Gold            │                      │                              │
└───────────────────────────────┬─────────────────────────────────────────────────┘
                                │
         ┌──────────────────────┼──────────────────────┐
         ▼                      ▼                      ▼
┌─────────────────┐  ┌─────────────────┐  ┌─────────────────────┐
│  BRONZE (raw)   │  │  SILVER (trat.)  │  │  GOLD (dim + fato)   │
│  7 tabelas      │  │  7 tabelas       │  │  6 dims + 10 fatos   │
│  append-only    │  │  deduplicado     │  │  prontos p/ consumo  │
│  lote_id + meta │  │  QUALIFY ROW_NUM │  │  métricas de negócio │
└────────┬────────┘  └────────┬────────┘  └──────────┬──────────┘
         │                    │                       │
         └────────────────────┼───────────────────────┘
                              ▼
┌─────────────────────────────────────────────────────────────────────────────────┐
│                        CAMADA DE QUALIDADE (DQ)                                  │
│  Notebooks de validação em cada camada (Bronze, Silver, Gold)                    │
│  Regras: duplicidade, nulos, integridade referencial, consistência              │
│  Persistência: workspace.monitoring.qualidade_resultados                        │
│  Gate bloqueante: falhas BLOQUEANTES interrompem o pipeline                     │
└───────────────────────────────┬─────────────────────────────────────────────────┘
                                │
                                ▼
┌─────────────────────────────────────────────────────────────────────────────────┐
│                    CAMADA DE MONITORAMENTO & OBSERVABILIDADE                     │
│  pipeline_runs      │  finops_ingestion    │  qualidade_resultados              │
│  (latência, status, │  (bytes BQ, custo,  │  (regras DQ, severidade,           │
│   volume, erros)    │   row_count)         │   gate bloqueante)                 │
│                     │                      │                                    │
│  ALERTAS: Databricks SQL Alerts → notificação por e-mail/webhook                │
│  DASHBOARDS: AI/BI Dashboards sobre workspace.monitoring                        │
└─────────────────────────────────────────────────────────────────────────────────┘
```

### 1.2 Princípios de Design

| Princípio | Descrição |
|---|---|
| **Imutabilidade da Bronze** | Dados brutos são append-only com `lote_id`, `data_ingestao`, `fonte` — nunca sobrescritos |
| **Gate de Qualidade** | Cada camada valida DQ antes de liberar a próxima; falhas bloqueantes interrompem o pipeline |
| **Observabilidade nativa** | Cada execução registra latência, volume, status e erros em tabelas Delta de monitoramento |
| **Idempotência** | Re-execuções não produzem duplicidade (Silver usa `QUALIFY ROW_NUMBER`, Gold usa `CREATE OR REPLACE`) |
| **Custo controlado** | `finops_ingestion` rastreia bytes processados/cobrados do BigQuery por carga |
| **Severidade graduada** | Regras DQ classificadas em BLOQUEANTE, ALTA, MÉDIA, BAIXA com ações distintas |

## 2. Stack Tecnológico

| Camada | Tecnologia | Função no Pipeline |
|---|---|---|
| **Fonte** | Google BigQuery (Base dos Dados) | Fonte externa — dados do INEP/ANA |
| **Conexão** | `google-cloud-bigquery` + service account GCP | Autenticação e consulta programática |
| **Processamento** | Databricks (PySpark + SQL Delta) | Transformações, DQ e carga em todas as camadas |
| **Armazenamento** | Delta Lake (Unity Catalog) | Tabelas ACID com time travel e schema evolution |
| **Orquestração** | AWS Step Functions + Databricks Jobs | 7 jobs sequenciais com dependência entre camadas |
| **Qualidade** | Notebooks PySpark com regras customizadas | Validação por camada com gate bloqueante |
| **Monitoramento** | Tabelas Delta em `workspace.monitoring` | `pipeline_runs`, `finops_ingestion`, `qualidade_resultados` |
| **Visualização** | AI/BI Dashboards (Lakeview) | Dashboards de DQ, latência, volume e FinOps |
| **Alertas** | Databricks SQL Alerts | Notificações automáticas por e-mail/webhook |
| **Catálogo** | Unity Catalog (`workspace` catalog) | Governança, ACLs e linhagem de dados |
| **Versionamento** | Git folder (opcional) | Controle de versão dos notebooks |

## 3. Requisitos de Qualidade de Dados

A qualidade de dados é implementada em **três camadas** (Bronze, Silver, Gold), cada uma com notebooks de validação que registram resultados na tabela `workspace.monitoring.qualidade_resultados`. O pipeline usa um sistema de **gate bloqueante**: regras classificadas como `BLOQUEANTE` interrompem a execução se falharem; regras `INFORMATIVA` ou `BAIXA` registram a falha mas permitem continuidade.

### 3.1 Tabela de Monitoramento de Qualidade

```sql
CREATE TABLE IF NOT EXISTS workspace.monitoring.qualidade_resultados (
    execution_id STRING,     -- UUID da execução DQ
    layer STRING,            -- BRONZE | SILVER | GOLD
    table_name STRING,       -- Tabela validada
    rule_name STRING,        -- Nome da regra (ex: CHAVE_DUPLICADA)
    severity STRING,         -- BLOQUEANTE | ALTA | MÉDIA | BAIXA | INFORMATIVA
    checked_at TIMESTAMP,    -- Momento da verificação
    failed_rows BIGINT,      -- Quantidade de registros com falha
    status STRING,           -- PASSOU | FALHOU
    details STRING            -- Contexto adicional (chaves, lote, etc.)
) USING DELTA
```

### 3.2 Modelo de Severidade

| Severidade | Ação | Exemplo de Uso |
|---|---|---|
| **BLOQUEANTE** | Interrompe o pipeline imediatamente | Chave nula, lote vazio, metadados nulos |
| **ALTA** | Registra + alerta, não bloqueia | Divergência de soma entre tabelas |
| **MÉDIA** | Registra + sinaliza para revisão | Variação de volume > 20% |
| **BAIXA** | Registra apenas | Duplicidade informativa na Bronze |
| **INFORMATIVA** | Registra contexto sem julgamento | Registros órfãos (sem meta) |

### 3.3 Verificação de Duplicidade

**Objetivo:** Identificar registros duplicados, definir regras de deduplicação e gerar métricas de ocorrências.

**Implementação atual:**
- **Bronze:** Regra `CHAVE_DUPLICADA` verifica duplicidade da chave de negócio no último lote. Severidade `INFORMATIVA` (Bronze é append-only e histórica).
- **Silver:** Regra `CHAVE_DUPLICADA` com severidade `BLOQUEANTE`. A deduplicação é feita via `QUALIFY ROW_NUMBER() OVER (PARTITION BY chave ORDER BY data_ingestao DESC) = 1`.
- **Gold:** Regra `CHAVE_DUPLICADA` com severidade `BLOQUEANTE` em todas as tabelas fato.

**Chaves de negócio por tabela:**

| Tabela | Chave de Negócio |
|---|---|
| `uf` | ano + sigla_uf + serie + rede |
| `meta_alfabetizacao_brasil` | ano + rede |
| `meta_alfabetizacao_uf` | ano + sigla_uf + rede |
| `meta_alfabetizacao_municipio` | ano + id_municipio |
| `municipio` | ano + id_municipio + serie + rede |
| `alunos` | ano + id_municipio + id_escola + id_aluno |
| `dicionario` | id_tabela + nome_coluna + chave |

**Exemplo de consulta de detecção:**

```sql
-- Detecção de duplicidade por chave de negócio
SELECT
    ano, sigla_uf, serie, rede_codigo,
    COUNT(*) AS total_ocorrencias
FROM workspace.silver.uf
GROUP BY ano, sigla_uf, serie, rede_codigo
HAVING COUNT(*) > 1
ORDER BY total_ocorrencias DESC
```

**Métricas geradas:** `failed_rows` (número de grupos duplicados) e `details` (chaves afetadas).

**Deduplicação na Silver (padrão aplicado):**

```sql
CREATE OR REPLACE TABLE workspace.silver.uf AS
SELECT * FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY ano, sigla_uf, serie, rede_codigo
            ORDER BY data_ingestao DESC
        ) AS rn
    FROM workspace.bronze.uf_raw
) WHERE rn = 1
```

> A estratégia de deduplicação mantém o registro mais recente (`data_ingestao DESC`) quando há múltiplos lotes para a mesma chave de negócio.

### 3.4 Detecção de Valores Ausentes

**Objetivo:** Identificar campos nulos ou vazios, aplicar regras de tratamento (preenchimento, descarte ou sinalização) e registrar a quantidade de inconsistências.

**Implementação atual:**
- **Bronze:** Regra `METADADOS_NULOS` (BLOQUEANTE) verifica se `lote_id`, `data_ingestao`, `fonte` são nulos. Regra `CHAVE_NULA` (BLOQUEANTE) verifica se qualquer coluna da chave de negócio é nula.
- **Silver:** Regra `CHAVE_NULA` (BLOQUEANTE) em todas as 6 tabelas principais.
- **Gold:** Regra `CHAVE_NULA` (BLOQUEANTE) em todas as tabelas fato.

**Estratégias de tratamento por camada:**

| Camada | Estratégia | Justificativa |
|---|---|---|
| Bronze | **Sinalização** — registra falha mas não altera dados brutos | Bronze é imutável e append-only |
| Silver | **Descarte/Rejeição** — registros com chaves nulas são bloqueantes | Silver é a camada confiável para análise |
| Gold | **Descarte/Rejeição** — chaves nulas são bloqueantes | Gold é a camada de consumo final |

**Exemplo de detecção de nulos em campos não-chave (extensão proposta):**

```sql
-- Detecção de valores ausentes em campos analíticos
SELECT
    'taxa_aprovacao' AS campo,
    COUNT(*) AS total_nulos
FROM workspace.silver.municipio
WHERE taxa_aprovacao IS NULL

UNION ALL

SELECT
    'taxa_reprovacao' AS campo,
    COUNT(*) AS total_nulos
FROM workspace.silver.municipio
WHERE taxa_reprovacao IS NULL
```

**Regra DQ para registro de valores ausentes (extensão proposta):**

```python
# Exemplo: detecção de nulos em colunas analíticas
regras_nulos = [
    ("municipio", "taxa_aprovacao", "ALTA"),
    ("municipio", "taxa_reprovacao", "MÉDIA"),
    ("alunos", "categoria_alfabetizacao", "BLOQUEANTE"),
]

for tabela, coluna, severidade in regras_nulos:
    count = spark.sql(f"""
        SELECT COUNT(*) AS n
        FROM workspace.silver.{tabela}
        WHERE `{coluna}` IS NULL
    """).first()["n"]

    registrar_resultado(
        f"workspace.silver.{tabela}",
        f"VALOR_AUSENTE_{coluna.upper()}",
        severidade,
        count,
        f"Coluna: {coluna}"
    )
```

**Tratamento de preenchimento (Silver → Gold):**

> **Regra:** Nulo nunca vira zero. A ausência de meta ou taxa deve permanecer como `NULL` e ser classificada explicitamente (ex.: `SEM_META`, `SEM_RESULTADO`). Preencher com `0.0` mascara a ausência como valor real e distorce indicadores como a taxa de cumprimento da meta.

### 3.5 Validação de Chaves de Relacionamento

**Objetivo:** Garantir integridade referencial entre tabelas, identificar registros órfãos e registrar falhas de relacionamento.

**Implementação atual (Silver):**

A validação de integridade referencial já é implementada no notebook `Silver/Qualidade_dados/01_validar_qualidade_silver` com duas regras `INFORMATIVA`:

1. **`RELACIONAMENTO_SEM_META_REDE_MUNICIPAL`** — Verifica resultados da rede Municipal (código '3') sem meta correspondente (ano + município + rede).
2. **`RELACIONAMENTO_SEM_MUNICIPIO`** — Verifica alunos sem município correspondente (ano + município).

**Exemplo da regra já implementada:**

```sql
-- Resultados municipais da rede pública sem meta correspondente
SELECT COUNT(*) AS total_sem_meta
FROM workspace.silver.municipio r
LEFT ANTI JOIN workspace.silver.meta_alfabetizacao_municipio m
  ON r.ano = m.ano
 AND r.id_municipio = m.id_municipio
 AND r.rede_codigo = m.rede_codigo
WHERE r.rede_codigo = '3'
```

```sql
-- Alunos sem município correspondente (por ano)
SELECT
    a.ano,
    COUNT(*) AS total_sem_municipio
FROM workspace.silver.alunos a
LEFT ANTI JOIN (
    SELECT DISTINCT ano, id_municipio
    FROM workspace.silver.municipio
) m
  ON a.ano = m.ano
 AND a.id_municipio = m.id_municipio
GROUP BY a.ano
ORDER BY a.ano
```

**Resultados reais da última execução (out/2026):**

| Relacionamento | Registros Órfãos | Severidade |
|---|---|---|
| municipio × meta (rede municipal) | 242 | INFORMATIVA |
| alunos × municipio | 3.209 | INFORMATIVA |

> A ausência de meta **não significa meta = zero** — a fonte de metas cobre 5.352 municípios por ano. Registros órfãos são sinalizados para investigação, não excluídos.

**Extensões propostas:**

```sql
-- Validar que todo id_municipio na Silver existe na dimensão dim_municipio do Gold
SELECT COUNT(*) AS dim_municipio_orfaos
FROM workspace.silver.municipio s
LEFT ANTI JOIN workspace.gold.dim_municipio d
  ON s.id_municipio = d.id_municipio
```

```sql
-- Validar que toda sigla_uf na Silver existe na dimensão dim_uf do Gold
SELECT COUNT(*) AS uf_orfaos
FROM (
    SELECT DISTINCT sigla_uf FROM workspace.silver.uf
) s
LEFT ANTI JOIN workspace.gold.dim_uf d
  ON s.sigla_uf = d.sigla_uf
```

### 3.6 Consistência Entre Tabelas

**Objetivo:** Validar regras de negócio entre diferentes conjuntos de dados, detectar divergências de valores, formatos ou estruturas e produzir relatórios de inconsistências.

**Regras de consistência implementadas e propostas:**

#### 3.6.1 Consistência de Schema (já implementada via Bronze)

A Bronze usa `mergeSchema = true` para tolerar evolução de schema da fonte. A Silver valida tipos com `CAST` explícito em cada coluna, garantindo que divergências de formato sejam tratadas.

```sql
-- Silver aplica casts explícitos (exemplo do tratamento UF)
CREATE OR REPLACE TABLE workspace.silver.uf AS
SELECT
    CAST(ano AS INT) AS ano,
    CAST(sigla_uf AS STRING) AS sigla_uf,
    CAST(serie AS STRING) AS serie,
    CAST(rede AS STRING) AS rede_codigo,
    CAST(taxa_aprovacao AS DOUBLE) AS taxa_aprovacao,
    CAST(taxa_reprovacao AS DOUBLE) AS taxa_reprovacao,
    CAST(taxa_abandono AS DOUBLE) AS taxa_abandono
FROM (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY ano, sigla_uf, serie, rede
            ORDER BY data_ingestao DESC
        ) AS rn
    FROM workspace.bronze.uf_raw
) WHERE rn = 1
```

#### 3.6.2 Consistência de Valores — Meta vs Resultado (já implementada no Gold)

O fato `ft_meta_vs_resultado_municipio` cruza metas com resultados municipais. A consistência é garantida pela construção do fato:

```sql
-- ft_meta_vs_resultado_municipio (simplificado)
SELECT
    r.ano,
    r.id_municipio,
    r.serie,
    r.rede_codigo,
    r.taxa_aprovacao AS resultado,
    m.meta_alfabetizacao AS meta,
    CASE
        WHEN r.taxa_aprovacao >= m.meta_alfabetizacao THEN 'META_ATINGIDA'
        ELSE 'META_NAO_ATINGIDA'
    END AS situacao_meta
FROM workspace.silver.municipio r
INNER JOIN workspace.silver.meta_alfabetizacao_municipio m
  ON r.ano = m.ano
 AND r.id_municipio = m.id_municipio
 AND r.rede_codigo = m.rede_codigo
WHERE r.rede_codigo = '3'
```

#### 3.6.3 Consistência de Amplitude de Valores (extensão proposta)

```sql
-- Validar que taxas estão entre 0 e 100
SELECT
    'TAXA_FORA_INTERVALO' AS regra,
    COUNT(*) AS total_inconsistente
FROM workspace.silver.municipio
WHERE taxa_aprovacao < 0 OR taxa_aprovacao > 100
   OR taxa_reprovacao < 0 OR taxa_reprovacao > 100
   OR taxa_abandono < 0 OR taxa_abandono > 100
```

#### 3.6.4 Consistência de Total — Brasil vs UF vs Município (extensão proposta)

```sql
-- Verificar divergência entre soma de UFs e total Brasil
SELECT
    ABS(
        (SELECT SUM(taxa_aprovacao) FROM workspace.silver.meta_alfabetizacao_uf WHERE ano = 2016)
        -
        (SELECT taxa_aprovacao FROM workspace.silver.meta_alfabetizacao_brasil WHERE ano = 2016)
    ) AS divergencia
```

> Esta regra deve ser `INFORMATIVA` pois a soma de taxas estaduais não é esperada que iguale a taxa nacional (médias ponderadas, não somas diretas).

### 3.7 Gate de Qualidade — Fluxo de Decisão

```
┌───────────────────────────────────────────────────────┐
│              EXECUÇÃO DQ POR CAMADA                     │
│                                                       │
│  Para cada tabela:                                    │
│    1. Identificar último lote válido (Bronze)          │
│    2. Executar regras DQ (nulos, duplicidade, etc.)   │
│    3. Persistir resultado em qualidade_resultados      │
│    4. Avaliar severidade:                              │
│       ├── BLOQUEANTE + FALHOU → raise RuntimeError    │
│       ├── ALTA + FALHOU → alertar (não bloqueia)       │
│       ├── MÉDIA/BAIXA/INFORMATIVA → apenas registrar   │
│    5. Gate final:                                      │
│       ├── Há bloqueantes? → PIPELINE INTERROMPIDO      │
│       └── Sem bloqueantes? → GATE LIBERADO ✅          │
└───────────────────────────────────────────────────────┘
```

**Fluxo de orquestração entre camadas:**

```
Job 01 (Ingestão Bronze)
  ↓ (sucesso)
Job 02 (DQ Bronze)  ← Gate bloqueante
  ↓ (aprovado)
Job 03 (Tratamento Silver)
  ↓ (sucesso)
Job 04 (DQ Silver)  ← Gate bloqueante
  ↓ (aprovado)
Job 05 (Dimensões Gold)
  ↓ (sucesso)
Job 06 (Fatos Gold)
  ↓ (sucesso)
Job 07 (DQ Gold)    ← Gate bloqueante
  ↓ (aprovado)
PIPELINE CONCLUÍDO ✅
```

Se qualquer DQ bloqueante falhar, o AWS Step Functions captura a exceção e não aciona os jobs subsequentes.

## 4. Requisitos de Monitoramento e Observabilidade

O monitoramento é implementado nativamente no pipeline via **três tabelas Delta** no schema `workspace.monitoring`, populadas automaticamente em cada execução. A observabilidade cobre quatro dimensões: falhas de ingestão, latência, volume de dados e alertas.

### 4.1 Tabelas de Monitoramento

#### 4.1.1 `pipeline_runs` — Execuções do Pipeline

```sql
CREATE TABLE IF NOT EXISTS workspace.monitoring.pipeline_runs (
    run_id STRING,            -- UUID da execução
    layer STRING,             -- BRONZE | SILVER | GOLD
    entity STRING,            -- Entidade processada (ex: uf, municipio, alunos)
    started_at TIMESTAMP,     -- Início da execução
    finished_at TIMESTAMP,    -- Fim da execução
    duration_seconds DOUBLE,  -- Duração em segundos
    status STRING,            -- SUCESSO | FALHA
    row_count BIGINT,         -- Registros processados
    error_message STRING      -- Mensagem de erro (se houver)
) USING DELTA
```

Cada chamada de `ingest_bq_table()` e cada notebook de tratamento/DQ registra uma linha — tanto em sucesso quanto em falha.

#### 4.1.2 `finops_ingestion` — Custo e Volume da Ingestão

```sql
CREATE TABLE IF NOT EXISTS workspace.monitoring.finops_ingestion (
    run_id STRING,            -- UUID da execução
    entity STRING,            -- Entidade ingerida
    source_table STRING,      -- Tabela fonte no BigQuery
    bytes_processed BIGINT,   -- Bytes processados pelo BigQuery
    bytes_billed BIGINT,      -- Bytes cobrados pelo BigQuery
    row_count BIGINT,         -- Registros ingeridos
    recorded_at TIMESTAMP     -- Momento do registro
) USING DELTA
```

#### 4.1.3 `qualidade_resultados` — Resultados das Regras DQ

```sql
CREATE TABLE IF NOT EXISTS workspace.monitoring.qualidade_resultados (
    execution_id STRING,      -- UUID da execução DQ
    layer STRING,             -- BRONZE | SILVER | GOLD
    table_name STRING,        -- Tabela validada
    rule_name STRING,         -- Nome da regra
    severity STRING,          -- BLOQUEANTE | ALTA | MÉDIA | BAIXA | INFORMATIVA
    checked_at TIMESTAMP,     -- Momento da verificação
    failed_rows BIGINT,       -- Registros com falha
    status STRING,            -- PASSOU | FALHOU
    details STRING             -- Contexto adicional
) USING DELTA
```

### 4.2 Falhas de Ingestão

**Objetivo:** Detectar erros durante a captura ou carregamento dos dados, registrar logs detalhados e acionar alertas automáticos para incidentes críticos.

**Implementação atual:**

A função `ingest_bq_table()` no notebook `_common_ingestao` já implementa detecção e registro de falhas:

```python
# Trecho do _common_ingestao (simplificado)
try:
    # ... ingestão BigQuery -> gravação Bronze ...
    
    # Registra SUCESSO
    spark.createDataFrame([
        (run_id, "BRONZE", entity, started, finished, 
         duration, "SUCESSO", row_count, None)
    ], schema).write.format("delta").mode("append")\
        .saveAsTable("workspace.monitoring.pipeline_runs")

except Exception as exc:
    finished = datetime.now(timezone.utc).replace(tzinfo=None)
    duration = time.perf_counter() - tick

    # Registra FALHA
    spark.createDataFrame([
        (run_id, "BRONZE", entity, started, finished,
         duration, "FALHA", row_count, str(exc)[:2000])
    ], schema).write.format("delta").mode("append")\
        .saveAsTable("workspace.monitoring.pipeline_runs")

    # IMPORTANTE: não engole o erro
    raise
```

**Pontos-chave:**
- `error_message` armazena até 2.000 caracteres da exceção
- `status = "FALHA"` permite filtrar falhas rapidamente
- O erro é propagado via `raise` para o orquestrador (AWS Step Functions)
- Cada falha tem `run_id` rastreável até o `lote_id` na Bronze

**Consulta de monitoramento de falhas:**

```sql
-- Falhas de ingestão nos últimos 7 dias
SELECT
    run_id,
    entity,
    started_at,
    duration_seconds,
    row_count,
    error_message
FROM workspace.monitoring.pipeline_runs
WHERE status = 'FALHA'
  AND started_at >= DATE_SUB(CURRENT_DATE(), 7)
ORDER BY started_at DESC
```

**Alerta automático (Databricks SQL Alert):**

```sql
-- Query base para alerta: falhas críticas de ingestão
SELECT COUNT(*) AS falhas_criticas
FROM workspace.monitoring.pipeline_runs
WHERE status = 'FALHA'
  AND started_at >= DATE_SUB(CURRENT_DATE(), 1)
```
> Condição do alerta: `falhas_criticas > 0` -> notificar por e-mail/webhook

### 4.3 Latência da Pipeline

**Objetivo:** Monitorar o tempo de processamento entre ingestão e disponibilização dos dados, definir indicadores de desempenho (KPIs e SLAs) e gerar alertas quando limites forem ultrapassados.

**Implementação atual:**

A tabela `pipeline_runs` já captura `duration_seconds` para cada execução. O campo é calculado com `time.perf_counter()` (precisão de microssegundos).

**KPIs e SLAs definidos:**

| Métrica | Definição | SLA | Alerta |
|---|---|---|---|
| **Latência de Ingestão** | Tempo da consulta BigQuery -> gravação Bronze | < 300s por tabela | > 600s |
| **Latência de Tratamento** | Tempo de leitura Bronze -> gravação Silver | < 180s por tabela | > 360s |
| **Latência de DQ** | Tempo de execução das regras DQ | < 120s por camada | > 300s |
| **Latência Total do Pipeline** | Soma de todos os jobs (Job 01 -> 07) | < 30 min | > 60 min |

**Consulta de latência por entidade:**

```sql
-- Latência média por entidade nos últimos 30 dias
SELECT
    entity,
    layer,
    COUNT(*) AS execucoes,
    ROUND(AVG(duration_seconds), 1) AS latencia_media_seg,
    ROUND(MAX(duration_seconds), 1) AS latencia_max_seg,
    ROUND(MIN(duration_seconds), 1) AS latencia_min_seg
FROM workspace.monitoring.pipeline_runs
WHERE status = 'SUCESSO'
  AND started_at >= DATE_SUB(CURRENT_DATE(), 30)
GROUP BY entity, layer
ORDER BY latencia_media_seg DESC
```

**Alerta de SLA excedido:**

```sql
-- Entidades com latência acima do SLA
SELECT
    entity,
    layer,
    duration_seconds,
    started_at
FROM workspace.monitoring.pipeline_runs
WHERE status = 'SUCESSO'
  AND started_at >= DATE_SUB(CURRENT_DATE(), 1)
  AND (
    (layer = 'BRONZE' AND duration_seconds > 600)
    OR (layer = 'SILVER' AND duration_seconds > 360)
    OR (layer = 'GOLD' AND duration_seconds > 300)
 )
ORDER BY duration_seconds DESC
```

### 4.4 Volume de Dados Processados

**Objetivo:** Acompanhar registros processados por execução, identificar variações anormais de volume e exibir métricas históricas e tendências.

**Implementação atual:**

A tabela `pipeline_runs` armazena `row_count` por execução. A tabela `finops_ingestion` também armazena `row_count` e `bytes_processed`/`bytes_billed`.

**Consulta de volume histórico:**

```sql
-- Volume processado por dia e camada
SELECT
    DATE(started_at) AS data_execucao,
    layer,
    COUNT(*) AS num_execucoes,
    SUM(row_count) AS total_registros,
    ROUND(AVG(row_count), 0) AS media_registros
FROM workspace.monitoring.pipeline_runs
WHERE status = 'SUCESSO'
  AND started_at >= DATE_SUB(CURRENT_DATE(), 90)
GROUP BY DATE(started_at), layer
ORDER BY data_execucao DESC, layer
```

**Detecção de variação anormal de volume:**

```sql
-- Variação percentual vs média histórica (janela de 30 dias)
WITH historico AS (
    SELECT
        entity,
        layer,
        AVG(row_count) AS media_hist,
        STDDEV(row_count) AS desvio_hist
    FROM workspace.monitoring.pipeline_runs
    WHERE status = 'SUCESSO'
      AND started_at >= DATE_SUB(CURRENT_DATE(), 30)
    GROUP BY entity, layer
),
ultima AS (
    SELECT
        entity,
        layer,
        row_count AS ultima_execucao
    FROM workspace.monitoring.pipeline_runs r1
    WHERE r1.started_at = (
        SELECT MAX(r2.started_at)
        FROM workspace.monitoring.pipeline_runs r2
        WHERE r2.entity = r1.entity
          AND r2.layer = r1.layer
          AND r2.status = 'SUCESSO'
    )
)
SELECT
    h.entity,
    h.layer,
    h.media_hist,
    u.ultima_execucao,
    ROUND(
        ((u.ultima_execucao - h.media_hist) / NULLIF(h.media_hist, 0)) * 100,
        1
    ) AS variacao_pct
FROM historico h
JOIN ultima u ON h.entity = u.entity AND h.layer = u.layer
WHERE ABS(
    ((u.ultima_execucao - h.media_hist) / NULLIF(h.media_hist, 0)) * 100
) > 20
ORDER BY ABS(variacao_pct) DESC
```

> Alerta sugerido: `variacao_pct` superior a +/-20% dispara notificação para investigação.

**Dashboard de FinOps — custo por ingestão:**

```sql
-- Custo acumulado do BigQuery por entidade
SELECT
    entity,
    COUNT(*) AS cargas,
    SUM(bytes_billed) AS total_bytes_cobrados,
    ROUND(SUM(bytes_billed) / 1024 / 1024 / 1024, 2) AS total_gb_cobrados,
    SUM(row_count) AS total_registros
FROM workspace.monitoring.finops_ingestion
GROUP BY entity
ORDER BY total_gb_cobrados DESC
```

### 4.5 Alertas de Erro

**Objetivo:** Configurar notificações automáticas para falhas críticas, definir níveis de severidade (baixo, médio, alto e crítico) e integrar com ferramentas de comunicação e observabilidade.

#### 4.5.1 Matriz de Severidade de Alertas

| Nível | Critério | Ação | Canal |
|---|---|---|---|
| **CRÍTICO** | DQ bloqueante falhou OU ingestão falhou | Notificação imediata + interrompe pipeline | E-mail + Webhook (Slack/Teams) |
| **ALTO** | Latência > 2x SLA OU variação de volume > 50% | Notificação em até 1h | E-mail |
| **MÉDIO** | DQ informativa falhou OU latência > 1.5x SLA | Notificação em até 4h | E-mail (resumo diário) |
| **BAIXO** | Variação de volume 20-50% | Registro no dashboard | Dashboard apenas |

#### 4.5.2 Alertas no Databricks (SQL Alerts)

**Alerta 1 — Falha crítica de DQ bloqueante:**

```sql
SELECT COUNT(*) AS dq_bloqueante_falhas
FROM workspace.monitoring.qualidade_resultados
WHERE severity = 'BLOQUEANTE'
  AND status = 'FALHOU'
  AND checked_at >= DATE_SUB(CURRENT_DATE(), 1)
```
> Condição: `dq_bloqueante_falhas > 0` -> **CRÍTICO**

**Alerta 2 — Falha de ingestão:**

```sql
SELECT COUNT(*) AS falhas_ingestao
FROM workspace.monitoring.pipeline_runs
WHERE status = 'FALHA'
  AND started_at >= DATE_SUB(CURRENT_DATE(), 1)
```
> Condição: `falhas_ingestao > 0` -> **CRÍTICO**

**Alerta 3 — Latência acima do SLA:**

```sql
SELECT COUNT(*) AS sla_violacoes
FROM workspace.monitoring.pipeline_runs
WHERE status = 'SUCESSO'
  AND started_at >= DATE_SUB(CURRENT_DATE(), 1)
  AND (
    (layer = 'BRONZE' AND duration_seconds > 600)
    OR (layer = 'SILVER' AND duration_seconds > 360)
    OR (layer = 'GOLD' AND duration_seconds > 300)
 )
```
> Condição: `sla_violacoes > 0` -> **ALTO**

**Alerta 4 — Variação anormal de volume:**

```sql
SELECT COUNT(*) AS variacoes_anormais
FROM (
    SELECT
        entity,
        row_count,
        AVG(row_count) OVER (PARTITION BY entity ORDER BY started_at
            ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING) AS media_previa
    FROM workspace.monitoring.pipeline_runs
    WHERE status = 'SUCESSO'
      AND started_at >= DATE_SUB(CURRENT_DATE(), 7)
) t
WHERE media_previa > 0
  AND ABS((row_count - media_previa) / media_previa) > 0.20
```
> Condição: `variacoes_anormais > 0` -> **MÉDIO**

#### 4.5.3 Integração com Ferramentas Externas

| Ferramenta | Tipo de Integração | Uso |
|---|---|---|
| **E-mail** | Databricks SQL Alert (nativo) | Notificação padrão para todos os níveis |
| **Slack/Teams** | Webhook URL no alerta | Notificação instantânea para CRÍTICO e ALTO |
| **AWS SNS** | Step Functions -> SNS Topic | Notificação de falha do orquestrador |
| **AI/BI Dashboard** | Lakeview sobre `workspace.monitoring` | Visualização contínua de métricas |
| **Databricks Logs** | Cluster logs + Job run logs | Diagnóstico detalhado de erros |

## 5. Resumo das Métricas de Monitoramento

### 5.1 Catálogo de Métricas

| Categoria | Métrica | Tabela Fonte | Query de Cálculo |
|---|---|---|---|
| **Falhas** | Total de falhas de ingestão (24h) | `pipeline_runs` | `COUNT(*) WHERE status='FALHA'` |
| **Falhas** | Total de DQ bloqueante falhou (24h) | `qualidade_resultados` | `COUNT(*) WHERE severity='BLOQUEANTE' AND status='FALHOU'` |
| **Latência** | Duração média por camada (30d) | `pipeline_runs` | `AVG(duration_seconds) GROUP BY layer` |
| **Latência** | P95 de duração por entidade | `pipeline_runs` | `PERCENTILE(duration_seconds, 0.95)` |
| **Latência** | Violações de SLA (24h) | `pipeline_runs` | `COUNT(*) WHERE duration > threshold` |
| **Volume** | Registros processados por dia | `pipeline_runs` | `SUM(row_count) GROUP BY DATE(started_at)` |
| **Volume** | Variação percentual vs média histórica | `pipeline_runs` | `(last - avg) / avg * 100` |
| **Volume** | Bytes cobrados no BigQuery | `finops_ingestion` | `SUM(bytes_billed)` |
| **Volume** | GB processados por entidade | `finops_ingestion` | `SUM(bytes_processed) / 1024^3` |
| **DQ** | Taxa de aprovação de regras DQ | `qualidade_resultados` | `SUM(status='PASSOU') / COUNT(*) * 100` |
| **DQ** | Registros com falha por regra | `qualidade_resultados` | `SUM(failed_rows) GROUP BY rule_name` |
| **DQ** | Registros órfãos por relacionamento | `qualidade_resultados` | `failed_rows WHERE rule_name LIKE 'RELACIONAMENTO%'` |

### 5.2 Dashboard Sugerido (AI/BI Lakeview)

**Painel 1 — Saúde do Pipeline (tempo real):**
- KPI: Sucesso vs Falha (últimas 24h)
- Timeline: Execuções por camada ao longo do tempo
- Tabela: Últimas 10 execuções com status e duração

**Painel 2 — Qualidade de Dados:**
- KPI: Taxa de aprovação DQ por camada
- Gráfico de barras: Falhas por regra DQ
- Tabela: Registros órfãos por relacionamento

**Painel 3 — Performance e SLA:**
- Gráfico de linha: Latência média por dia e camada
- Medidor: % de execuções dentro do SLA
- Tabela: Top 5 entidades com maior latência

**Painel 4 — FinOps:**
- KPI: GB cobrados no BigQuery (mês atual)
- Gráfico de barras: Custo por entidade
- Tabela: Detalhamento de bytes por carga

## 6. Melhores Práticas

### 6.1 Qualidade de Dados

| Prática | Implementação no Projeto |
|---|---|
| **Validação em cada camada** | Notebooks DQ separados para Bronze, Silver e Gold |
| **Gate bloqueante** | Regras BLOQUEANTES interrompem o pipeline via `raise RuntimeError` |
| **Severidade graduada** | 5 níveis: BLOQUEANTE, ALTA, MÉDIA, BAIXA, INFORMATIVA |
| **Rastreabilidade** | `execution_id` + `lote_id` + `run_id` conectam DQ, Bronze e monitoramento |
| **Idempotência** | Silver usa `QUALIFY ROW_NUMBER`; Gold usa `CREATE OR REPLACE` |
| **Não excluir sem investigar** | Registros órfãos são INFORMATIVA — não descarte automático |
| **Contexto nos detalhes** | Campo `details` registra chaves, lote e contagem afetada |
| **Evolução de schema** | Bronze com `mergeSchema=true`; Silver com `CAST` explícito |

### 6.2 Observabilidade

| Prática | Implementação no Projeto |
|---|---|
| **Monitoramento nativo** | Cada execução registra em `pipeline_runs` (sucesso e falha) |
| **Logs estruturados** | `error_message` truncado em 2.000 chars para garantir persistência |
| **Propagação de erro** | `raise` após registrar — nunca engolir exceção |
| **FinOps integrado** | `finops_ingestion` rastreia bytes e custo do BigQuery por carga |
| **Histórico preservado** | Tabelas Delta de monitoramento são append-only com time travel |
| **Alertas automatizados** | SQL Alerts do Databricks sobre `workspace.monitoring` |
| **Visualização contínua** | Dashboards AI/BI (Lakeview) sobre tabelas de monitoramento |

### 6.3 Orquestração

| Prática | Implementação no Projeto |
|---|---|
| **Dependência entre camadas** | Job N só executa se Job N-1 + DQ N-1 passaram |
| **AWS Step Functions** | Máquina pai (Map sequencial sobre a lista de Jobs) + máquina filha (run-now e polling do status) — `orquestracao/*.asl.json` |
| **Falha** | Job sem `SUCCESS` → estado `Fail` → cadeia interrompida; jobs seguintes não rodam |
| **Retries** | Retry apenas em chamadas HTTP transitórias (429/503); falha de Job/DQ **não** é reexecutada automaticamente |
| **Notificação de falha** | EventBridge (FAILED/TIMED_OUT/ABORTED) → SNS → e-mail: `orquestracao/alerta_falha.cfn.yaml` (**implantação a confirmar**) |

### 6.4 Governança

| Prática | Implementação no Projeto |
|---|---|
| **Unity Catalog** | Tabelas em `workspace.bronze`, `workspace.silver`, `workspace.gold` |
| **Linhagem de dados** | Unity Catalog rastreia linhagem entre Bronze -> Silver -> Gold |
| **Acesso controlado** | Permissões via Unity Catalog (GRANT SELECT/USE SCHEMA) |
| **Versionamento** | Notebooks em Git folder (opcional) para controle de versão |

### 6.5 Recomendações de Evolução

1. **Databricks SQL Alerts:** Configurar os 4 alertas descritos na seção 4.5 com cron de avaliação a cada 1h
2. **AI/BI Dashboard:** Criar dashboard Lakeview sobre `workspace.monitoring` com os 4 painéis da seção 5.2
3. **Alertas de consistência adicionais:** Implementar regras de amplitude de valores (seção 3.6.3) e validação cruzada Brasil vs UF (seção 3.6.4)
4. **Detecção de nulos em colunas analíticas:** Implementar regras `VALOR_AUSENTE` para colunas não-chave (seção 3.4)
5. **Validação referencial Gold:** Adicionar regras de integridade entre Silver e dimensões Gold (seção 3.5)
6. **Data Quality Monitoring:** Ativar DQM nativo do Databricks para tabelas Gold com monitors automáticos de freshness, completeness e null constraints
7. **Retry de Jobs:** decidir se falhas transitórias de infraestrutura merecem retry; falha de DQ nunca deve ser reexecutada às cegas
8. **Webhook Slack/Teams:** Configurar webhook URL nos SQL Alerts para notificação instantânea de incidentes CRÍTICOS

## 7. Mapa de Cobertura dos Requisitos

### 7.1 Requisitos de Qualidade de Dados

| Requisito | Status | Onde | Notebook/Arquivo |
|---|---|---|---|
| **1. Duplicidade** | ✅ Implementado | Bronze (informativa), Silver (bloqueante), Gold (bloqueante) | `Bronze/Qualidade_dados/01_validar_qualidade_bronze`, `Silver/Qualidade_dados/01_validar_qualidade_silver`, `Gold/Qualidade_dados/01_validar_qualidade_gold` |
| **2. Valores ausentes** | ✅ Parcial | Bronze e Silver (chaves), extensão proposta para colunas analíticas | Notebooks DQ de cada camada |
| **3. Chaves de relacionamento** | ✅ Implementado | Silver com LEFT ANTI JOIN (municipio x meta, alunos x municipio) | `Silver/Qualidade_dados/01_validar_qualidade_silver` |
| **4. Consistência entre tabelas** | ✅ Parcial | Casts na Silver, meta vs resultado no Gold; extensões propostas para amplitude e totais | `Silver/Tratamento/*`, `Gold/Fato/01_construir_fatos` |

### 7.2 Requisitos de Monitoramento e Observabilidade

| Requisito | Status | Onde | Implementação |
|---|---|---|---|
| **1. Falhas de ingestão** | ✅ Implementado | `pipeline_runs` com status=FALHA + error_message | `_common_ingestao` (try/except + raise) |
| **2. Latência** | ✅ Implementado | `pipeline_runs` com `duration_seconds` | `time.perf_counter()` por execução |
| **3. Volume processado** | ✅ Implementado | `pipeline_runs.row_count` + `finops_ingestion` | Registro automático por carga |
| **4. Alertas de erro** | ⚠️ A confirmar | Consultas em `Monitoring/alertas_sql.sql`; alertas existem no workspace conforme prints, mas a evidência de disparo real ainda precisa ser registrada | `Monitoring/alertas_sql.sql`, `Gold/Qualidade_dados/02_teste_falha_dq_demo` |

### 7.3 Legenda

- ✅ Implementado: já existe no pipeline atual
- ⚠️ A confirmar: código entregue, mas sem evidência de execução no ambiente real
- Extensões propostas: documentadas neste notebook com exemplos de código prontos para implementação